#Environment Setting

In [ ]:
import os
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
path = "/content/drive/MyDrive/UTP_model_training"
os.chdir(path)
print(os.getcwd())

In [ ]:
!pip install openai-clip
!pip install transformers==4.47.1 numpy==1.26.4

In [ ]:
import transformers
print(transformers.__version__)
import numpy as np
print(np.__version__)

# Util Functions

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

# Load the CLIP model (ViT-L/14) and processor from Hugging Face
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() # 纯提取特征，确保模型处于 eval 模式

input_dim = 1536 # 768 (image) + 768 (text)

# 确保保存模型的文件夹存在
os.makedirs(os.path.join(path, "candidate_models"), exist_ok=True)

# Function to process the image
def encode_image(image_path):
    # Load and preprocess the image
    image = Image.open(image_path)
    image_inputs = processor(images=image, return_tensors="pt")

    # Move image input to device (CPU/GPU)
    image_inputs = {key: value.to(device) for key, value in image_inputs.items()}

    # Get image embeddings
    with torch.no_grad():
        image_embeddings = model.get_image_features(**image_inputs)

    # Normalize embeddings
    image_embeddings = image_embeddings / image_embeddings.norm(dim=-1, keepdim=True)

    return image_embeddings

# Function to process the text
def encode_text(json_path):
    text = None
    with open(json_path, 'r') as f:
        element_list = json.load(f)
    if isinstance(element_list, list):
        text = element_list[-1]["summarization"]
    else:
        text = element_list["compos"][-1]["summarization"]

    # Tokenize and preprocess the text
    text_inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True)

    # Move text input to device (CPU/GPU)
    text_inputs = {key: value.to(device) for key, value in text_inputs.items()}

    # Get text embeddings
    with torch.no_grad():
        text_embeddings = model.get_text_features(**text_inputs)

    # Normalize embeddings
    text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)

    return text_embeddings

# Encode image text
#image_features = encode_image("example1.jpg")
#text_features = encode_text("example1.json")
#print(image_features.shape)
#print(text_features.shape)

# Concatenate the features
#concat_features = torch.cat((image_features, text_features), dim=1)
#print(concat_features.shape)

class BinaryClsModel(nn.Module):
    def __init__(self):
        super(BinaryClsModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 128)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(128, 64)
        self.dropout3 = nn.Dropout(0.3)
        self.fc4 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = torch.relu(self.fc3(x))
        x = self.dropout3(x)
        x = torch.sigmoid(self.fc4(x))
        return x

#image text UI dataset
class UIDataset(Dataset):
    def __init__(self, image_pairs, labels):
        self.image_pairs = image_pairs
        self.labels = labels
        #self.preprocess = preprocess

    def __len__(self):
        return len(self.image_pairs)

    def __getitem__(self, idx):
        image1_path, json_path = self.image_pairs[idx]
        label = self.labels[idx]

        image = encode_image(image1_path)
        text = encode_text(json_path)

        #image1 = self.preprocess(Image.open(image1_path))
        #image2 = self.preprocess(Image.open(image2_path))
        #combined_features = torch.cat((image1, image2), dim=1)
        return image,text, label

def shuffle_two_lists_preserve_original(list1, list2):
    if len(list1) != len(list2):
        raise ValueError("两个列表的长度必须相等。")

    combined = list(zip(list1, list2))
    random.shuffle(combined)
    shuffled_list1, shuffled_list2 = zip(*combined)

    return list(shuffled_list1), list(shuffled_list2)

def k_fold_cross_validation(list1, list2, k=10):
    if len(list1) != len(list2):
        raise ValueError("两个列表的长度必须相等。")

    n = len(list1)
    fold_size = n // k
    folds = []

    # 因为输入已经是经过 shuffle_two_lists_preserve_original 打乱的，这里直接切片即可
    for i in range(k):
        start = i * fold_size
        end = (i + 1) * fold_size if i < k - 1 else n

        # 验证集切片
        val_set_input = list1[start:end]
        val_set_output = list2[start:end]

        # 训练集切片（拼接验证集前后的数据）
        train_set_input = list1[:start] + list1[end:]
        train_set_output = list2[:start] + list2[end:]

        folds.append(((train_set_input, train_set_output), (val_set_input, val_set_output)))

    return folds

#build image text - training set and validation set using hifi data
def build_train_val_set_with_hifi_data(fold_index):
    input_pairs = []
    labels = []

    with open("sk.json", "r") as f:
        sk = json.load(f)

    for directory , _, _  in os.walk('uncertain_data'):
        #print(directory)
        directory_name_list = directory.split('/')
        print(directory_name_list)

        try:
            directory_name = directory_name_list[1]
        except IndexError:
            continue
        start_img_name = directory_name+'start.jpg'
        end_json_name = 'hand_'+directory_name+'end.json'
        start_img_path = os.path.join(path,'uncertain_data',directory_name,start_img_name)
        end_json_path = os.path.join(path,'uncertain_data',directory_name,end_json_name)

        input_pairs.append((start_img_path,end_json_path))
        labels.append(1)  #0_weights 系列的setting是把ambiguous sample当成1
    print(len(input_pairs))
    print(len(labels))

    for directory , _, _  in os.walk('certain_data'):
        #print(directory)
        directory_name_list = directory.split('/')
        print(directory_name_list)
        try:
            directory_name = directory_name_list[1]
        except IndexError:
            continue
        start_img_name = directory_name+'start.jpg'
        end_json_name = 'YOLO_'+directory_name+'end.json'

        start_img_path = os.path.join(path,'certain_data',directory_name,start_img_name)
        end_json_path = os.path.join(path,'certain_data',directory_name,end_json_name)
        #print(start_img_path,end_img_path)
        input_pairs.append((start_img_path,end_json_path))
        labels.append(0)

    shuffled_pairs, shuffled_labels = shuffle_two_lists_preserve_original(input_pairs, labels)

    print(len(shuffled_pairs))
    print(len(shuffled_labels))

    folds = k_fold_cross_validation(shuffled_pairs, shuffled_labels)

    # return: training input, training_label, val input, val label
    return folds[fold_index][0][0], folds[fold_index][0][1], folds[fold_index][1][0], folds[fold_index][1][1]

#build image text - training and validation set- using lowfi data
def build_train_val_set_with_lowfi_data(fold_index):
    input_pairs = []
    labels = []

    with open("sk.json", "r") as f:
        sk = json.load(f)

    for directory , _, _  in os.walk('uncertain_data'):
        #print(directory)
        directory_name_list = directory.split('/')
        print(directory_name_list)

        try:
            directory_name = directory_name_list[1]
        except IndexError:
            continue
        #start_img_name = directory_name+'start.jpg'

        if directory_name in sk["sk"]:
            print("detect")
            #start_wf_path = os.path.join(path,'ambiguous',directory_name,start_img_name)
        else:
            start_wf_name = directory_name+'wireframe_start.jpg'
            start_wf_path = os.path.join(path,'uncertain_data',directory_name,start_wf_name)
            end_json_name = 'hand_'+directory_name+'end.json'
            end_json_path = os.path.join(path,'uncertain_data',directory_name,end_json_name)
            #print(start_img_path,end_img_path)
            input_pairs.append((start_wf_path,end_json_path))
            labels.append(1)  #0_weights 系列的setting是把ambiguous sample当成1
    print(len(input_pairs))
    print(len(labels))

    for directory , _, _  in os.walk('certain_data'):
        #print(directory)
        directory_name_list = directory.split('/')
        print(directory_name_list)
        try:
            directory_name = directory_name_list[1]
        except IndexError:
            continue
        start_wf_name = directory_name+'wireframe_start.jpg'
        end_json_name = 'YOLO_'+directory_name+'end.json'
        start_wf_path = os.path.join(path,'certain_data',directory_name,start_wf_name)
        end_json_path = os.path.join(path,'certain_data',directory_name,end_json_name)

        input_pairs.append((start_wf_path,end_json_path))
        labels.append(0)

    print(len(input_pairs))
    print(len(labels))

    shuffled_pairs, shuffled_labels = shuffle_two_lists_preserve_original(input_pairs, labels)

    print(len(shuffled_pairs))
    print(len(shuffled_labels))

    folds = k_fold_cross_validation(shuffled_pairs, shuffled_labels)

    # return: training input, training_label, val input, val label
    return folds[fold_index][0][0], folds[fold_index][0][1], folds[fold_index][1][0], folds[fold_index][1][1]

# merge hifi and lowfi data
def data_merge(hifi_input, hifi_label, lowfi_input, lowfi_label):
    total_input = hifi_input + lowfi_input
    total_labels = hifi_label + lowfi_label
    print("total_input len", len(total_input))
    print("total_output len", len(total_labels))
    shuffled_input, shuffled_labels = shuffle_two_lists_preserve_original(total_input, total_labels)

    return shuffled_input, shuffled_labels

def build_hifi_test_set():
    #build image text testing set
    #build hifi test set
    test_pairs = []
    test_labels = []
    for i in range(1,141):
        start_img_path = os.path.join(path,"benchmark_data","negative_data",str(i),str(i)+"start.jpg")
        #end_img_path = os.path.join(path, "test_finetune", str(i), str(i)+"wireframe_end.jpg")
        end_json_path = os.path.join(path, "benchmark_data","negative_data", str(i), "hand_" + str(i)+"end.json")
        if(os.path.exists(start_img_path) and os.path.exists(end_json_path)):
            #print(i)
            test_pairs.append((start_img_path,end_json_path))
            test_labels.append(0)
            #test_labels.append(1)

    for i in range(1,141):
        start_img_path = os.path.join(path,"benchmark_data","positive_data",str(i),str(i)+"start.jpg")
        #end_img_path = os.path.join(path,"test_ambigu_finetune",str(i)+"wireframe_end.jpg")
        end_json_path = os.path.join(path, "benchmark_data","positive_data",str(i), "hand_" + str(i)+"end.json")
        if(os.path.exists(start_img_path) and os.path.exists(end_json_path)):
            #print(i)
            test_pairs.append((start_img_path,end_json_path))
            test_labels.append(1) #0_weights 系列的setting是把ambiguous sample当成1
            #test_labels.append(0) 0_model_weights 系列的setting是把ambiguous sample当成0

    print(len(test_pairs))
    print(len(test_labels))
    return test_pairs, test_labels

def build_lowfi_test_set():
    #build image text testing set
    #build wireframe test set
    test_pairs = []
    test_labels = []
    for i in range(1,141):
        start_img_path = os.path.join(path,"benchmark_data","negative_data",str(i),str(i)+"_start_mockup.jpg")
        #end_img_path = os.path.join(path, "test_finetune", str(i), str(i)+"wireframe_end.jpg")
        end_json_path = os.path.join(path, "benchmark_data","negative_data", str(i), "hand_" + str(i)+"end.json")
        if(os.path.exists(start_img_path) and os.path.exists(end_json_path)):
            test_pairs.append((start_img_path,end_json_path))
            test_labels.append(0)
            #test_labels.append(1)

    for i in range(1,141):
        start_img_path = os.path.join(path,"benchmark_data","positive_data",str(i),str(i)+"_start_mockup.png")
        #end_img_path = os.path.join(path,"test_ambigu_finetune",str(i)+"wireframe_end.jpg")
        end_json_path = os.path.join(path,"benchmark_data","positive_data",str(i), "hand_" + str(i)+"end.json")
        if(os.path.exists(start_img_path) and os.path.exists(end_json_path)):
            #print(i)
            test_pairs.append((start_img_path,end_json_path))
            test_labels.append(1) #0_weights 系列的setting是把ambiguous sample当成1
            #test_labels.append(0) 0_model_weights 系列的setting是把ambiguous sample当成0

    print(len(test_pairs))
    print(len(test_labels))

    return test_pairs, test_labels

def evaluate_model(model, dataloader, criterion, device):
    """在验证集上评估模型，返回平均 Loss、F1-Score、Accuracy、Precision、Recall"""
    model.eval()  # 切换到评估模式（关闭 Dropout）
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():  # 验证阶段不需要计算梯度
        for img_feat, txt_feat, labels in dataloader:
            img_feat, txt_feat, labels = img_feat.to(device), txt_feat.to(device), labels.to(device)

            combined_features = torch.cat((img_feat, txt_feat), dim=2)

            print(combined_features.shape)

            outputs = model(combined_features).squeeze()

            # 前向传播
            #outputs = model(img_feat, txt_feat).squeeze(-1)

            # 计算当前 Batch 的 Loss
            loss = criterion(outputs, labels.float())
            val_loss += loss.item() * img_feat.size(0)

            # 将概率转换为二分类标签（阈值 0.5）
            preds = (outputs >= 0.5).float().cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    # 计算全局各项指标
    avg_loss = val_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)

    return avg_loss, acc, precision, recall, f1

import os
import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

def evaluate_ensemble_model(model, candidate_model_dir, weight_name, dataloader, device):
    # 1. 提取测试集里所有的真实标签（用于最后对齐计算指标）
    all_labels = []
    for _, _, labels in dataloader:
        all_labels.extend(labels.numpy())
    all_labels = np.array(all_labels)

    # 2. 准备盛放 10 个模型预测结果的“大总管”列表
    all_fold_probs = []

    # 实例化一个干净的模型用于轮流加载权重
    model.eval()  # 必须处于 eval 模式

    # 3. 轮流让 10 个模型对测试集进行全量预测
    with torch.no_grad():
        for fold in range(10):
            weight_path = os.path.join(path,candidate_model_dir, f"{fold}{weight_name}")
            if not os.path.exists(weight_path):
                raise FileNotFoundError(f"找不到第 {fold} 折的模型权重: {weight_path}")

            model.load_state_dict(torch.load(weight_path, map_location=device))

            fold_probs = []
            # 遍历测试集 DataLoader 提取当前模型的概率
            for img_feat, txt_feat, _ in dataloader:
                img_feat, txt_feat = img_feat.to(device), txt_feat.to(device)

                combined_features = torch.cat((img_feat, txt_feat), dim=2)
                print(combined_features.shape)
                outputs = model(combined_features).squeeze()

                fold_probs.extend(outputs.cpu().numpy())

            all_fold_probs.append(fold_probs)

    final_probs = np.mean(all_fold_probs, axis=0)

    # 将平均概率转换为二分类预测标签（阈值 0.5）
    final_preds = (final_probs >= 0.5).astype(float)

    # 计算分类各项核心指标
    acc = accuracy_score(all_labels, final_preds)
    f1 = f1_score(all_labels, final_preds, zero_division=0)
    precision = precision_score(all_labels, final_preds, zero_division=0)
    recall = recall_score(all_labels, final_preds, zero_division=0)

    return acc, precision, recall, f1

def calculate_inverted_metrics(N, accuracy, precision, recall):
    # 边界情况处理：如果原本就是 100% 完美的模型
    if precision == 1.0 and recall == 1.0:
        if accuracy == 1.0:
            return {"new_precision": 1.0, "new_recall": 1.0, "msg": "原模型完美，对调后亦完美"}
        else:
            raise ValueError("数据不逻辑：Precision和Recall为1时，Accuracy必须为1。")

    # 1. 计算中间系数 k (即 TP 占总样本 N 的比例)
    # 公式分母: (1/P) + (1/R) - 2
    denominator = (1.0 / precision) + (1.0 / recall) - 2.0

    if abs(denominator) < 1e-9:
        # 当 P + R = 2*P*R 时的特殊平衡状态处理
        k = (1.0 - accuracy) / 1e-9
    else:
        k = (1.0 - accuracy) / denominator

    # 2. 还原四民矩阵的比例/数量
    tp = k * N
    fp = k * ((1.0 - precision) / precision) * N
    fn = k * ((1.0 - recall) / recall) * N
    tn = (accuracy * N) - tp

    # 四舍五入还原最接近的整数样本数
    tp_count = round(tp)
    fp_count = round(fp)
    fn_count = round(fn)
    tn_count = round(tn)

    print(f"--- 还原原始混淆矩阵 (约件数) ---")
    print(f"原 TP: {tp_count}, 原 FP: {fp_count}")
    print(f"原 FN: {fn_count}, 原 TN: {tn_count}")
    print(f"验证还原总数: {tp_count + fp_count + fn_count + tn_count} (输入 N: {N})")
    print(f"--------------------------------")

    # 3. 计算标签对调后的新指标
    # 新 Precision = TN / (TN + FN)
    # 新 Recall = TN / (TN + FP)

    # 使用原始 float 比例计算，避免四舍五入带来的微小误差
    new_precision = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    new_recall = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "new_precision": round(new_precision, 4),
        "new_recall": round(new_recall, 4)
    }

def k_fold_dataset_split(fold_index):
    # prepare hifi training data
    hifi_traning_input, hifi_training_label, hifi_val_input, hifi_val_label = build_train_val_set_with_hifi_data(fold_index)
    hifi_training_dataset = UIDataset(hifi_traning_input, hifi_training_label)
    hifi_training_dataloader = DataLoader(hifi_training_dataset, batch_size=256, shuffle=True)
    print(len(hifi_traning_input),len(hifi_training_label))

    # prepare hifi+lowfi training data
    lowfi_traning_input, lowfi_training_label, lowfi_val_input,lowfi_val_label = build_train_val_set_with_lowfi_data(fold_index)
    print(len(lowfi_traning_input),len(lowfi_training_label))

    total_input, total_labels = data_merge(hifi_traning_input, hifi_training_label, lowfi_traning_input, lowfi_training_label)
    total_training_dataset = UIDataset(total_input, total_labels)
    total_training_dataloader = DataLoader(total_training_dataset, batch_size=256, shuffle=True)

    # prepare hifi val data
    hifi_val_dataset = UIDataset(hifi_val_input, hifi_val_label)
    print(hifi_val_input[0])
    hifi_val_dataloader = DataLoader(hifi_val_dataset, batch_size=32, shuffle=False)

    # prepare lowfi val data
    lowfi_val_dataset = UIDataset(lowfi_val_input, lowfi_val_label)
    print(lowfi_val_input[0])
    lowfi_val_dataloader = DataLoader(lowfi_val_dataset, batch_size=32, shuffle=False)

    return hifi_training_dataloader, hifi_val_dataloader, lowfi_val_dataloader, total_training_dataloader


# Training

In [ ]:
print("=== 开始训练 ===")

for k in range (1,10):
    hifi_training_dataloader, hifi_val_dataloader, lowfi_val_dataloader, total_training_dataloader = k_fold_dataset_split(k)
    bcmodel = BinaryClsModel().to(device)
    optimizer = optim.Adam(bcmodel.parameters(), lr=1e-3)
    criterion = torch.nn.BCELoss()
    num_epochs = 50

    best_hifi_f1 = 0.0 # 记录最高的hifi F1-score
    best_lowfi_f1 = 0.0 #记录最高的lowfi F1-score

    if not os.path.exists(os.path.join(path, "candidate_models")):
        os.makedirs(os.path.join(path, "candidate_models"))

    for epoch in range(num_epochs):
        bcmodel.train()  # 确保切换回训练模式（开启 Dropout）
        train_loss = 0.0


        for img_feat, txt_feat, labels in hifi_training_dataloader:
            img_feat, txt_feat, labels = img_feat.to(device), txt_feat.to(device), labels.to(device)

            combined_features = torch.cat((img_feat, txt_feat), dim=2)
            print(combined_features.shape)
            outputs = bcmodel(combined_features).squeeze()

            loss = criterion(outputs, labels.float())

            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * img_feat.size(0)

        avg_train_loss = train_loss / len(hifi_training_dataloader.dataset)

        # 每个 Epoch 结束时跑一次验证集
        val_loss, val_acc, precision, recall, val_f1 = evaluate_model(bcmodel, hifi_val_dataloader, criterion, device)

        print(f"Epoch {epoch+1:02d}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"Val precision: {precision:.4f} | "
              f"Val recall: {recall:.4f} | "
              f"Val F1: {val_f1:.4f}")

        # 推荐最优hifi模型
        if val_f1 > best_hifi_f1:
            best_hifi_f1 = val_f1
            print(f"{epoch}_weights_model1.pth, 当前最高 hifi F1: {best_hifi_f1:.4f}")
            #torch.save(bcmodel.state_dict(), os.path.join(path, "candidate_models", f"{epoch}_weights_model1.pth"))
            torch.save(bcmodel.state_dict(), os.path.join(path, "candidate_models", f"{k}_best_weights_model1_hifi.pth"))


        val_loss, val_acc, precision, recall, val_f1= evaluate_model(bcmodel, lowfi_val_dataloader, criterion, device)

        print(f"Epoch {epoch+1:02d}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"Val precision: {precision:.4f} | "
              f"Val recall: {recall:.4f} | "
              f"Val F1: {val_f1:.4f}")

        # 推荐最优lowfi模型
        if val_f1 > best_hifi_f1:
            best_hifi_f1 = val_f1
            print(f"{epoch}_weights_model1.pth, 当前最高 lowfi F1: {best_hifi_f1:.4f}")
            #torch.save(bcmodel.state_dict(), os.path.join(path, "candidate_models", f"{epoch}_weights_model1.pth"))
            torch.save(bcmodel.state_dict(), os.path.join(path, "candidate_models", f"{k}_best_weights_model1_lowfi.pth"))




# Testing



In [ ]:
#prepare hifi test data
hifi_test_pairs, hifi_test_labels=build_hifi_test_set()
hifi_test_dataset = UIDataset(hifi_test_pairs, hifi_test_labels)
hifi_test_dataloader = DataLoader(hifi_test_dataset, batch_size=32, shuffle=False)

#prepare lowfi test data
lowfi_test_pairs, lowfi_test_labels=build_lowfi_test_set()
lowfi_test_dataset = UIDataset(lowfi_test_pairs, lowfi_test_labels)
lowfi_test_dataloader = DataLoader(lowfi_test_dataset, batch_size=32, shuffle=False)

#initialize model and criterion
bcmodel = BinaryClsModel().to(device)
criterion = torch.nn.BCELoss()

In [ ]:
# Test using one selected model
bcmodel_path= os.path.join(path, "examplary_models_for_inference", "train_hybrid_infer_mockup.pth")
bcmodel.load_state_dict(torch.load(bcmodel_path))
test_loss, test_acc, test_precision, test_recall, test_f1 = evaluate_model(bcmodel, dataloader, criterion, device)
print(f"final test | "
          f"Acc: {test_acc:.4f} | "
          f"precision: {test_precision:.4f} | "
          f"recall: {test_recall:.4f} | "
          f"F1: {test_f1:.4f}")

N=140
output = calculate_inverted_metrics(N, test_acc, test_precision, test_recall)
print("precision on negavie samples", output["new_precision"], "recall on negative samples", output["new_recall"])